In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import geopandas as gpd
from sklearn.model_selection import train_test_split

In [2]:
df_property = pd.read_parquet('../data/processed/properties_clean.parquet')

df_property

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10661,"Flat 2B, Avenue Studios, Sydney Close, London,...",SW3 6HW,1750000,2024-04-05,51.491681,-0.172813,1.0,1.0,1.0,120.0,Leasehold,Terraced,E,14.375126
10662,"99 Victorian Grove, London, N16 8EJ",N16 8EJ,580000,2024-09-13,51.557519,-0.076808,2.0,1.0,1.0,89.0,Leasehold,Terraced,D,13.270783
10663,"9A Cambray Road, London, SW12 0DX",SW12 0DX,266092,2024-07-24,51.444387,-0.140667,2.0,2.0,1.0,78.0,Leasehold,Terraced,C,12.491597
10664,"20 Barfett Street, London, W10 4NP",W10 4NP,715000,2024-09-04,51.527446,-0.206082,2.0,1.0,1.0,67.0,Freehold,Terraced Bungalow,D,13.480038


In [3]:
df_property['total_rooms'] = df_property['bedrooms'] + df_property['bathrooms'] + df_property['livingRooms']

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0


In [4]:
df_property['bath_bed_ratio'] = df_property['bathrooms'] / df_property['bedrooms'].replace(0, 1)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5


In [5]:
df_property['area_per_room_sqm'] = df_property['floorAreaSqM'] / df_property['total_rooms']

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000


In [6]:
df_property['is_freehold'] = (df_property['tenure'] == 'Freehold').astype(int)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333,1
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333,1
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000,1
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000,1
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000,1


In [7]:
energy_map = {
    'A': 7,
    'B': 6,
    'C': 5,
    'D': 4,
    'E': 3,
    'F': 2,
    'G': 1
}

df_property['energy_score'] = df_property['currentEnergyRating'].map(energy_map)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333,1,3
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333,1,5
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000,1,4
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000,1,3
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000,1,4


In [8]:
CENTRE_LAT = 51.50734
CENTRE_LONG = -0.12765
EARTH_RADIUS = 6371

def haversine_distance(lat1, long1, lat2, long2, radius):
    lat1, long1, lat2, long2 = map(np.radians, [lat1, long1, lat2, long2]) # Converting degrees into radians

    lat_distance = lat2 - lat1
    long_distance = long2 - long1

    angular_separation = np.sin(lat_distance / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(long_distance / 2)**2 # Calculating the haversine formula component measuring the angular separation between two points on a spherical surface

    central_angle = 2 * np.arcsin(np.sqrt(angular_separation)) # Calculating the central angle between the two points

    return radius * central_angle

In [9]:
df_property['distance_to_centre_km'] = haversine_distance(df_property['latitude'], df_property['longitude'], CENTRE_LAT, CENTRE_LONG, EARTH_RADIUS)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.0,12.333333,1,3,4.172360
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.0,20.333333,1,5,6.992492
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.5,16.250000,1,4,8.344587
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.5,19.250000,1,3,11.435198
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.5,20.750000,1,4,6.087600


In [10]:
df_stops = pd.read_parquet('../data/processed/stops_clean.parquet')

df_stops

,atcocode,commonname,latitude,longitude,stoptype,town
0,0170ZZAVBIT0,Bitton (Avon Valley Railway),51.430350,-2.476210,TMU,Unknown
1,0170ZZAVOLD0,Oldland (Avon Valley Railway),51.443660,-2.467350,TMU,Unknown
2,0400ZZLUAMS,Amersham Underground Station,51.674198,-0.607434,TMU,Unknown
3,0400ZZLUAMS0,Amersham Underground Station,51.674206,-0.607362,TMU,Unknown
4,0400ZZLUCAL,Chalfont & Latimer Underground Station,51.668075,-0.560495,TMU,Unknown
...,...,...,...,...,...,...
5120,9400ZZTWWJS,Balfour Street (Edinburgh Trams),55.965364,-3.176258,MET,Unknown
5121,9400ZZTWWJT,McDonald Road (Edinburgh Trams),55.960678,-3.181465,MET,Unknown
5122,9400ZZTWWJU,Picardy Place (Edinburgh Trams),55.956817,-3.186794,MET,Unknown
5123,9400ZZBPTAL,Talbot Road (Blackpool Tramway),53.819144,-3.054507,MET,Unknown


In [11]:
df_metro = df_stops[df_stops['stoptype'] == 'MET']

df_metro

,atcocode,commonname,latitude,longitude,stoptype,town
46,1500ZZLUBKH0,Buckhurst Hill Underground Station,51.626625,0.046498,MET,Buckhurst Hill
47,1500ZZLUBKH1,Buckhurst Hill Underground Station,51.625231,0.047042,MET,Buckhurst Hill
48,1500ZZLUBKH2,Buckhurst Hill Underground Station,51.625318,0.046179,MET,Buckhurst Hill
49,1500ZZLUCWL0,Chigwell Underground Station,51.617856,0.074258,MET,Chigwell
50,1500ZZLUCWL1,Chigwell Underground Station,51.617905,0.075026,MET,Chigwell
...,...,...,...,...,...,...
5120,9400ZZTWWJS,Balfour Street (Edinburgh Trams),55.965364,-3.176258,MET,Unknown
5121,9400ZZTWWJT,McDonald Road (Edinburgh Trams),55.960678,-3.181465,MET,Unknown
5122,9400ZZTWWJU,Picardy Place (Edinburgh Trams),55.956817,-3.186794,MET,Unknown
5123,9400ZZBPTAL,Talbot Road (Blackpool Tramway),53.819144,-3.054507,MET,Unknown


In [12]:
df_rail = df_stops[df_stops['stoptype'] == 'RLY']

df_rail

,atcocode,commonname,latitude,longitude,stoptype,town
1459,9100ABDARE,Aberdare Rail Station,51.715058,-3.443083,RLY,Unknown
1460,9100ASHYDN,Ashley Down,51.478750,-2.576700,RLY,Unknown
1461,9100SWNACFC,Corfe Castle,50.638300,-2.055100,RLY,Unknown
1462,9100BEULYPK,Beaulieu Park,51.758057,0.518336,RLY,Unknown
1463,9100ELINTN,East Linton,55.985600,-2.657900,RLY,Unknown
...,...,...,...,...,...,...
4144,9100ZZTYLIN,Lintley Rail Station,54.853553,-2.488311,RLY,Unknown
4145,9100ZZTYSGY,Slaggyford Rail Station,54.865492,-2.506747,RLY,Unknown
4146,9100PTWYPR,Portway Park & Ride Rail Station,51.488108,-2.687559,RLY,Unknown
4147,9100LEVEN,Leven Rail Station,56.192386,-3.001894,RLY,Unknown


In [13]:
property_coordinates = np.radians(df_property[['latitude', 'longitude']].values)
metro_coordinates = np.radians(df_metro[['latitude', 'longitude']].values)
rail_coordinates = np.radians(df_rail[['latitude', 'longitude']].values)

In [14]:
metro_tree = BallTree(metro_coordinates, metric='haversine')

In [15]:
rail_tree = BallTree(rail_coordinates, metric='haversine')

In [16]:
metro_distances, _ = metro_tree.query(property_coordinates, k=1)

df_property['distance_to_nearest_metro_km'] = metro_distances.flatten() * EARTH_RADIUS

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km,distance_to_nearest_metro_km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,Bungalow Property,E,13.071070,3.0,1.0,12.333333,1,3,4.172360,1.192172
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,Bungalow Property,C,12.971540,3.0,1.0,20.333333,1,5,6.992492,3.057838
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,Bungalow Property,D,13.142166,4.0,0.5,16.250000,1,4,8.344587,3.340976
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,Bungalow Property,E,12.936034,4.0,0.5,19.250000,1,3,11.435198,1.951058
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,Bungalow Property,D,13.972514,4.0,0.5,20.750000,1,4,6.087600,0.416076


In [17]:
rail_distances, _ = rail_tree.query(property_coordinates, k=1)

df_property['distance_to_nearest_rail_km'] = rail_distances.flatten() * EARTH_RADIUS

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km,distance_to_nearest_metro_km,distance_to_nearest_rail_km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,E,13.071070,3.0,1.0,12.333333,1,3,4.172360,1.192172,1.117444
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,C,12.971540,3.0,1.0,20.333333,1,5,6.992492,3.057838,0.942559
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,D,13.142166,4.0,0.5,16.250000,1,4,8.344587,3.340976,0.617187
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,E,12.936034,4.0,0.5,19.250000,1,3,11.435198,1.951058,1.424745
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,D,13.972514,4.0,0.5,20.750000,1,4,6.087600,0.416076,0.378393


In [18]:
DISTANCE_RADIUS = 1
DISTANCE_RADIUS_RAD = DISTANCE_RADIUS / EARTH_RADIUS

In [19]:
df_property['total_metro_within_1km'] = metro_tree.query_radius(property_coordinates, r=DISTANCE_RADIUS_RAD, count_only=True)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km,distance_to_nearest_metro_km,distance_to_nearest_rail_km,total_metro_within_1km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,13.071070,3.0,1.0,12.333333,1,3,4.172360,1.192172,1.117444,0
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,12.971540,3.0,1.0,20.333333,1,5,6.992492,3.057838,0.942559,0
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,13.142166,4.0,0.5,16.250000,1,4,8.344587,3.340976,0.617187,0
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,12.936034,4.0,0.5,19.250000,1,3,11.435198,1.951058,1.424745,0
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,13.972514,4.0,0.5,20.750000,1,4,6.087600,0.416076,0.378393,3


In [20]:
df_property[['distance_to_nearest_metro_km', 'distance_to_nearest_rail_km', 'total_metro_within_1km']].describe()

,distance_to_nearest_metro_km,distance_to_nearest_rail_km,total_metro_within_1km
count,10666.000000,10666.000000,10666.000000
mean,1.216364,0.836448,1.387680
std,1.029516,0.548794,1.801961
min,0.026543,0.015380,0.000000
25%,0.455564,0.452295,0.000000
50%,0.885805,0.719626,1.000000
75%,1.683891,1.083935,2.000000
max,6.669956,3.751706,11.000000


In [21]:
df_schools = pd.read_parquet('../data/processed/schools_clean.parquet')

df_schools

,URN,Postcode,As at date,Phase,Overall effectiveness,latitude,longitude
0,100154.0,SE3 9GJ,2020-08-31,Primary,2.0,51.461191,0.022444
1,100197.0,SE28 8AS,2020-08-31,Primary,4.0,51.505195,0.111098
2,100378.0,W12 0NW,2020-08-31,Special,1.0,51.507064,-0.242606
3,100743.0,SE6 3QW,2020-08-31,Secondary,3.0,51.423142,-0.020763
4,100802.0,SE5 8SN,2020-08-31,Primary,2.0,51.469523,-0.087106
...,...,...,...,...,...,...,...
23919,151093.0,BD20 7JS,2024-08-31,Primary,2.0,53.893729,-1.991202
23920,151109.0,SN4 0EJ,2024-08-31,Primary,2.0,51.547482,-1.696835
23921,151130.0,SN25 1JP,2024-08-31,Primary,2.0,51.592445,-1.810736
23922,151198.0,GU3 2HS,2024-08-31,Primary,4.0,51.256221,-0.683754


In [22]:
df_good_schools = df_schools[df_schools['Overall effectiveness'].isin([1, 2])]

df_good_schools

,URN,Postcode,As at date,Phase,Overall effectiveness,latitude,longitude
0,100154.0,SE3 9GJ,2020-08-31,Primary,2.0,51.461191,0.022444
2,100378.0,W12 0NW,2020-08-31,Special,1.0,51.507064,-0.242606
4,100802.0,SE5 8SN,2020-08-31,Primary,2.0,51.469523,-0.087106
8,101017.0,SW12 9SS,2020-08-31,Primary,2.0,51.441690,-0.152648
11,102197.0,HA8 5RU,2020-08-31,Primary,1.0,51.599933,-0.279886
...,...,...,...,...,...,...,...
23917,151005.0,PR3 3BJ,2024-08-31,Primary,2.0,53.808687,-2.612114
23918,151084.0,PR2 2SA,2024-08-31,Primary,2.0,53.765942,-2.726346
23919,151093.0,BD20 7JS,2024-08-31,Primary,2.0,53.893729,-1.991202
23920,151109.0,SN4 0EJ,2024-08-31,Primary,2.0,51.547482,-1.696835


In [23]:
school_coordinates = np.radians(df_schools[['latitude', 'longitude']].values)
good_school_coordinates = np.radians(df_good_schools[['latitude', 'longitude']].values)

In [24]:
school_tree = BallTree(school_coordinates, metric='haversine')

In [25]:
good_school_tree = BallTree(good_school_coordinates, metric='haversine')

In [26]:
good_school_distances, _ = good_school_tree.query(property_coordinates, k=1)

df_property['distance_to_nearest_good_school_km'] = good_school_distances.flatten() * EARTH_RADIUS

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km,distance_to_nearest_metro_km,distance_to_nearest_rail_km,total_metro_within_1km,distance_to_nearest_good_school_km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,3.0,1.0,12.333333,1,3,4.172360,1.192172,1.117444,0,0.285644
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,3.0,1.0,20.333333,1,5,6.992492,3.057838,0.942559,0,0.161388
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,4.0,0.5,16.250000,1,4,8.344587,3.340976,0.617187,0,0.448822
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,4.0,0.5,19.250000,1,3,11.435198,1.951058,1.424745,0,0.137802
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,4.0,0.5,20.750000,1,4,6.087600,0.416076,0.378393,3,0.133251


In [27]:
df_property['total_good_Schools_within_1km'] = good_school_tree.query_radius(property_coordinates, r=DISTANCE_RADIUS_RAD, count_only=True)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km,distance_to_nearest_metro_km,distance_to_nearest_rail_km,total_metro_within_1km,distance_to_nearest_good_school_km,total_good_Schools_within_1km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,1.0,12.333333,1,3,4.172360,1.192172,1.117444,0,0.285644,11
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,1.0,20.333333,1,5,6.992492,3.057838,0.942559,0,0.161388,8
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,0.5,16.250000,1,4,8.344587,3.340976,0.617187,0,0.448822,8
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,0.5,19.250000,1,3,11.435198,1.951058,1.424745,0,0.137802,2
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,0.5,20.750000,1,4,6.087600,0.416076,0.378393,3,0.133251,10


In [28]:
indices = school_tree.query_radius(property_coordinates, r=DISTANCE_RADIUS_RAD)

average_ofsted = []

for i in indices:
    if len(i) == 0:
        average_ofsted.append(np.nan)
    else:
        average_ofsted.append(df_schools.iloc[i]['Overall effectiveness'].mean())

df_property['average_ofsted_score_within_1km'] = average_ofsted

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,area_per_room_sqm,is_freehold,energy_score,distance_to_centre_km,distance_to_nearest_metro_km,distance_to_nearest_rail_km,total_metro_within_1km,distance_to_nearest_good_school_km,total_good_Schools_within_1km,average_ofsted_score_within_1km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,12.333333,1,3,4.172360,1.192172,1.117444,0,0.285644,11,1.363636
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,20.333333,1,5,6.992492,3.057838,0.942559,0,0.161388,8,1.888889
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,16.250000,1,4,8.344587,3.340976,0.617187,0,0.448822,8,1.875000
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,19.250000,1,3,11.435198,1.951058,1.424745,0,0.137802,2,2.000000
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,20.750000,1,4,6.087600,0.416076,0.378393,3,0.133251,10,1.400000


In [29]:
df_crime = pd.read_parquet('../data/processed/crime_clean.parquet')

df_crime

,month,latitude,longitude,crime_type
0,2023-01-01,51.493025,-0.139391,Anti-social behaviour
1,2023-01-01,51.506559,0.104245,Other theft
2,2023-01-01,51.505609,0.103426,Other theft
3,2023-01-01,51.506559,0.104245,Other theft
4,2023-01-01,51.505609,0.103426,Other theft
...,...,...,...,...
1975532,2024-09-01,51.546788,0.002760,Other theft
1975533,2024-09-01,51.548120,0.003265,Other theft
1975534,2024-09-01,51.546788,0.002760,Other theft
1975535,2024-09-01,51.547541,0.005014,Shoplifting


In [30]:
df_crime['geometry'] = gpd.points_from_xy(df_crime.longitude, df_crime.latitude)

df_crime.head()

,month,latitude,longitude,crime_type,geometry
0,2023-01-01,51.493025,-0.139391,Anti-social behaviour,POINT (-0.13939 51.49302)
1,2023-01-01,51.506559,0.104245,Other theft,POINT (0.10424 51.50656)
2,2023-01-01,51.505609,0.103426,Other theft,POINT (0.10343 51.50561)
3,2023-01-01,51.506559,0.104245,Other theft,POINT (0.10424 51.50656)
4,2023-01-01,51.505609,0.103426,Other theft,POINT (0.10343 51.50561)


In [31]:
crime_coordinates = np.radians(df_crime[['latitude', 'longitude']].values)

In [32]:
crime_tree = BallTree(crime_coordinates, metric='haversine')

In [33]:
crime_radius_list = [0.5, 1, 2]

for radius in crime_radius_list:
    radius_rad = radius / EARTH_RADIUS
    if radius < 1:
        df_property[f'total_crime_within_{int(radius * 1000)}m'] = crime_tree.query_radius(property_coordinates, r=radius_rad, count_only=True)
    else:
        df_property[f'total_crime_within_{radius}km'] = crime_tree.query_radius(property_coordinates, r=radius_rad, count_only=True)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,distance_to_centre_km,distance_to_nearest_metro_km,distance_to_nearest_rail_km,total_metro_within_1km,distance_to_nearest_good_school_km,total_good_Schools_within_1km,average_ofsted_score_within_1km,total_crime_within_500m,total_crime_within_1km,total_crime_within_2km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,4.172360,1.192172,1.117444,0,0.285644,11,1.363636,2012,8336,39417
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,6.992492,3.057838,0.942559,0,0.161388,8,1.888889,1027,3954,13959
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,8.344587,3.340976,0.617187,0,0.448822,8,1.875000,949,4519,13005
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,11.435198,1.951058,1.424745,0,0.137802,2,2.000000,299,1846,13887
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,6.087600,0.416076,0.378393,3,0.133251,10,1.400000,1897,11029,51898


In [34]:
df_crime['crime_type'].unique()

array(['Anti-social behaviour', 'Other theft', 'Public order',
       'Shoplifting', 'Vehicle crime', 'Violence and sexual offences',
       'Criminal damage and arson', 'Drugs', 'Burglary', 'Robbery',
       'Other crime', 'Possession of weapons', 'Bicycle theft',
       'Theft from the person'], dtype=object)

In [35]:
violent_crimes = ['Violence and sexual offences', 'Robbery', 'Possession of weapons']
property_crimes = ['Vehicle crime', 'Criminal damage and arson', 'Burglary']
theft_crimes = ['Other theft', 'Shoplifting', 'Bicycle theft', 'Theft from the person']
disorder_crimes = ['Anti-social behaviour', 'Public order', 'Drugs']

In [36]:
def crime_feature(df_crime, crime_category, radius):
    df_crime_category = df_crime[df_crime['crime_type'].isin(crime_category)]

    crime_category_coordinates = np.radians(df_crime_category[['latitude', 'longitude']].values)

    crime_category_tree = BallTree(crime_category_coordinates, metric='haversine')

    radius_rad = radius / EARTH_RADIUS

    return crime_category_tree.query_radius(property_coordinates, r=radius_rad, count_only=True)

In [37]:
df_property['total_violent_crimes_within_1km'] = crime_feature(df_crime, violent_crimes, DISTANCE_RADIUS)
df_property['total_property_crimes_within_1km'] = crime_feature(df_crime, property_crimes, DISTANCE_RADIUS)
df_property['total_theft_crimes_within_1km'] = crime_feature(df_crime, theft_crimes, DISTANCE_RADIUS)
df_property['total_disorder_crimes_within_1km'] = crime_feature(df_crime, disorder_crimes, DISTANCE_RADIUS)

df_property.head()

,fullAddress,postcode,history_price,history_date,latitude,longitude,bedrooms,bathrooms,livingRooms,floorAreaSqM,...,distance_to_nearest_good_school_km,total_good_Schools_within_1km,average_ofsted_score_within_1km,total_crime_within_500m,total_crime_within_1km,total_crime_within_2km,total_violent_crimes_within_1km,total_property_crimes_within_1km,total_theft_crimes_within_1km,total_disorder_crimes_within_1km
0,"Flat 49, Cadogan House, Beaufort Street, Londo...",SW3 5BL,475000,2024-07-19,51.483373,-0.174024,1.0,1.0,1.0,37.0,...,0.285644,11,1.363636,2012,8336,39417,1904,2217,2152,2027
1,"56B Silvester Road, London, SE22 9PB",SE22 9PB,430000,2024-07-11,51.454592,-0.072674,1.0,1.0,1.0,61.0,...,0.161388,8,1.888889,1027,3954,13959,983,1054,929,964
2,"66 Robson Road, London, SE27 9LB",SE27 9LB,510000,2024-04-12,51.434855,-0.096453,2.0,1.0,1.0,65.0,...,0.448822,8,1.875000,949,4519,13005,1336,1110,807,1219
3,"6 Glenwood Avenue, London, NW9 7PJ",NW9 7PJ,415000,2024-02-16,51.572604,-0.255432,2.0,1.0,1.0,77.0,...,0.137802,2,2.000000,299,1846,13887,523,421,302,587
4,"8 Hofland Road, London, W14 0LN",W14 0LN,1170000,2024-01-05,51.499899,-0.214786,2.0,1.0,1.0,83.0,...,0.133251,10,1.400000,1897,11029,51898,2806,1974,2417,3761


In [38]:
df_property.columns

Index(['fullAddress', 'postcode', 'history_price', 'history_date', 'latitude',
       'longitude', 'bedrooms', 'bathrooms', 'livingRooms', 'floorAreaSqM',
       'tenure', 'propertyType', 'currentEnergyRating', 'log_price',
       'total_rooms', 'bath_bed_ratio', 'area_per_room_sqm', 'is_freehold',
       'energy_score', 'distance_to_centre_km', 'distance_to_nearest_metro_km',
       'distance_to_nearest_rail_km', 'total_metro_within_1km',
       'distance_to_nearest_good_school_km', 'total_good_Schools_within_1km',
       'average_ofsted_score_within_1km', 'total_crime_within_500m',
       'total_crime_within_1km', 'total_crime_within_2km',
       'total_violent_crimes_within_1km', 'total_property_crimes_within_1km',
       'total_theft_crimes_within_1km', 'total_disorder_crimes_within_1km'],
      dtype='object')

In [39]:
df_model = df_property.drop(columns=['fullAddress', 'postcode', 'history_price', 'history_date', 'latitude', 'longitude', 'total_crime_within_500m', 'total_crime_within_2km'])

df_model

,bedrooms,bathrooms,livingRooms,floorAreaSqM,tenure,propertyType,currentEnergyRating,log_price,total_rooms,bath_bed_ratio,...,distance_to_nearest_rail_km,total_metro_within_1km,distance_to_nearest_good_school_km,total_good_Schools_within_1km,average_ofsted_score_within_1km,total_crime_within_1km,total_violent_crimes_within_1km,total_property_crimes_within_1km,total_theft_crimes_within_1km,total_disorder_crimes_within_1km
0,1.0,1.0,1.0,37.0,Freehold,Bungalow Property,E,13.071070,3.0,1.00,...,1.117444,0,0.285644,11,1.363636,8336,1904,2217,2152,2027
1,1.0,1.0,1.0,61.0,Freehold,Bungalow Property,C,12.971540,3.0,1.00,...,0.942559,0,0.161388,8,1.888889,3954,983,1054,929,964
2,2.0,1.0,1.0,65.0,Freehold,Bungalow Property,D,13.142166,4.0,0.50,...,0.617187,0,0.448822,8,1.875000,4519,1336,1110,807,1219
3,2.0,1.0,1.0,77.0,Freehold,Bungalow Property,E,12.936034,4.0,0.50,...,1.424745,0,0.137802,2,2.000000,1846,523,421,302,587
4,2.0,1.0,1.0,83.0,Freehold,Bungalow Property,D,13.972514,4.0,0.50,...,0.378393,3,0.133251,10,1.400000,11029,2806,1974,2417,3761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10661,1.0,1.0,1.0,120.0,Leasehold,Terraced,E,14.375126,3.0,1.00,...,1.658731,2,0.210871,10,1.500000,13028,2003,3264,5300,2415
10662,2.0,1.0,1.0,89.0,Leasehold,Terraced,D,13.270783,4.0,0.50,...,0.600511,0,0.389657,17,1.647059,10330,2739,1799,2331,3387
10663,2.0,2.0,1.0,78.0,Leasehold,Terraced,C,12.491597,5.0,1.00,...,0.825060,1,0.151880,10,1.500000,6030,1376,1491,1406,1723
10664,2.0,1.0,1.0,67.0,Freehold,Terraced Bungalow,D,13.480038,4.0,0.50,...,0.728981,2,0.224381,23,1.708333,13293,3892,2403,2188,4727


In [40]:
df_model = pd.get_dummies(df_model, columns=['tenure', 'propertyType', 'currentEnergyRating'], drop_first=True)

df_model.head()

,bedrooms,bathrooms,livingRooms,floorAreaSqM,log_price,total_rooms,bath_bed_ratio,area_per_room_sqm,is_freehold,energy_score,...,propertyType_Semi-Detached Property,propertyType_Terrace Property,propertyType_Terraced,propertyType_Terraced Bungalow,currentEnergyRating_B,currentEnergyRating_C,currentEnergyRating_D,currentEnergyRating_E,currentEnergyRating_F,currentEnergyRating_G
0,1.0,1.0,1.0,37.0,13.071070,3.0,1.0,12.333333,1,3,...,False,False,False,False,False,False,False,True,False,False
1,1.0,1.0,1.0,61.0,12.971540,3.0,1.0,20.333333,1,5,...,False,False,False,False,False,True,False,False,False,False
2,2.0,1.0,1.0,65.0,13.142166,4.0,0.5,16.250000,1,4,...,False,False,False,False,False,False,True,False,False,False
3,2.0,1.0,1.0,77.0,12.936034,4.0,0.5,19.250000,1,3,...,False,False,False,False,False,False,False,True,False,False
4,2.0,1.0,1.0,83.0,13.972514,4.0,0.5,20.750000,1,4,...,False,False,False,False,False,False,True,False,False,False


In [41]:
crime_features = ['total_crime_within_1km', 'total_violent_crimes_within_1km', 'total_property_crimes_within_1km', 'total_theft_crimes_within_1km', 'total_disorder_crimes_within_1km']

for feature in crime_features:
    df_model[feature] = np.log1p(df_model[feature])

df_model[crime_features]

,total_crime_within_1km,total_violent_crimes_within_1km,total_property_crimes_within_1km,total_theft_crimes_within_1km,total_disorder_crimes_within_1km
0,9.028459,7.552237,7.704361,7.674617,7.614805
1,8.282736,6.891626,6.961296,6.835185,6.872128
2,8.416267,7.198184,7.013016,6.694562,7.106606
3,7.521318,6.261492,6.045005,5.713733,6.376727
4,9.308374,7.939872,7.588324,7.790696,8.232706
...,...,...,...,...,...
10661,9.474933,7.602900,8.091015,8.575651,7.789869
10662,9.242904,7.915713,7.495542,7.754482,8.127995
10663,8.704668,7.227662,7.307873,7.249215,7.452402
10664,9.495068,8.266935,7.784889,7.691200,8.461258


In [42]:
df_model.isna().sum()

bedrooms                                0
bathrooms                               0
livingRooms                             0
floorAreaSqM                            0
log_price                               0
total_rooms                             0
bath_bed_ratio                          0
area_per_room_sqm                       0
is_freehold                             0
energy_score                            0
distance_to_centre_km                   0
distance_to_nearest_metro_km            0
distance_to_nearest_rail_km             0
total_metro_within_1km                  0
distance_to_nearest_good_school_km      0
total_good_Schools_within_1km           0
average_ofsted_score_within_1km        10
total_crime_within_1km                  0
total_violent_crimes_within_1km         0
total_property_crimes_within_1km        0
total_theft_crimes_within_1km           0
total_disorder_crimes_within_1km        0
tenure_Leasehold                        0
propertyType_Converted Flat       

In [43]:
df_model = df_model.dropna(subset=['average_ofsted_score_within_1km'])

df_model.shape

(10656, 47)

In [44]:
df_model.to_parquet('../data/modelling/complete_modelling_data.parquet')

In [45]:
X = df_model.drop(columns=['log_price'])
y = df_model['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [46]:
X_train.to_parquet('../data/modelling/X_train.parquet')
X_test.to_parquet('../data/modelling/X_test.parquet')
y_train.to_frame(name='log_price').to_parquet('../data/modelling/y_train.parquet')
y_test.to_frame(name='log_price').to_parquet('../data/modelling/y_test.parquet')